# 01 — SEC EDGAR Data Collection

Downloads 10-K and 10-Q annual/quarterly filings from SEC EDGAR for
US-listed companies. Extracts financial statements (Income Statement,
Balance Sheet, Cash Flow) as training data for zone classification
and SBERT fine-tuning.

**Target**: ~500 companies × 2 years = 1,000 filings

In [ ]:
# Cell 1: Install dependencies & configure
!pip install -q requests beautifulsoup4 lxml tqdm PyYAML PyMuPDF

import requests, json, time, os, re
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import yaml

# Mount Drive (Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Load config
CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

DATA_DIR = Path(cfg['drive']['data_dir'])
EDGAR_DIR = DATA_DIR / 'edgar'
EDGAR_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COMPANIES = cfg['data']['edgar_target_companies']  # 500
YEARS_PER_COMPANY = cfg['data']['edgar_years_per_company']  # 2

# SEC requires a User-Agent header with your identity
HEADERS = {
    'User-Agent': os.environ.get('EDGAR_USER_AGENT', 'Numera ML training@numera.ai'),
    'Accept-Encoding': 'gzip, deflate',
}
print(f'Config loaded — target: {TARGET_COMPANIES} companies, {YEARS_PER_COMPANY} years each')
print(f'Data dir: {EDGAR_DIR}')

In [ ]:
# Cell 2: Get S&P 500 CIK numbers from SEC company tickers
TICKERS_URL = 'https://www.sec.gov/files/company_tickers.json'

def fetch_cik_mapping(limit: int = 500) -> list[dict]:
    """Fetch CIK→ticker mappings from SEC."""
    resp = requests.get(TICKERS_URL, headers=HEADERS)
    resp.raise_for_status()
    data = resp.json()
    companies = []
    for entry in list(data.values())[:limit]:
        companies.append({
            'cik': str(entry['cik_str']).zfill(10),
            'ticker': entry['ticker'],
            'title': entry['title'],
        })
    return companies

companies = fetch_cik_mapping(TARGET_COMPANIES)
print(f'Fetched {len(companies)} companies')
companies[:5]

In [ ]:
# Cell 3: Search EDGAR EFTS for 10-K filings per company
EFTS_URL = 'https://efts.sec.gov/LATEST/search-index?q=%22annual+report%22&dateRange=custom'
FILING_URL = 'https://www.sec.gov/cgi-bin/browse-edgar'
SUBMISSIONS_URL = 'https://data.sec.gov/submissions/CIK{cik}.json'

def get_recent_filings(cik: str, form_type: str = '10-K', count: int = 2) -> list[dict]:
    """Get recent filings for a CIK from the submissions API."""
    url = SUBMISSIONS_URL.format(cik=cik)
    resp = requests.get(url, headers=HEADERS)
    if resp.status_code != 200:
        return []
    data = resp.json()
    recent = data.get('filings', {}).get('recent', {})
    forms = recent.get('form', [])
    accessions = recent.get('accessionNumber', [])
    dates = recent.get('filingDate', [])
    docs = recent.get('primaryDocument', [])
    
    results = []
    for i, form in enumerate(forms):
        if form == form_type and len(results) < count:
            results.append({
                'accession': accessions[i].replace('-', ''),
                'accession_raw': accessions[i],
                'date': dates[i],
                'document': docs[i],
                'cik': cik,
            })
    return results

# Test with first company
test = get_recent_filings(companies[0]['cik'])
print(f'{companies[0]["ticker"]}: {len(test)} filings found')
test

In [ ]:
# Cell 4: Download 10-K filing documents (HTML/XML)
ARCHIVES_URL = 'https://www.sec.gov/Archives/edgar/data/{cik}/{accession}/{doc}'

def download_filing(filing: dict, out_dir: Path) -> Path | None:
    """Download a single filing document."""
    url = ARCHIVES_URL.format(
        cik=filing['cik'].lstrip('0'),
        accession=filing['accession'],
        doc=filing['document'],
    )
    out_path = out_dir / f"{filing['cik']}_{filing['accession_raw']}.html"
    if out_path.exists():
        return out_path
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        out_path.write_bytes(resp.content)
        return out_path
    except Exception as e:
        print(f'  Failed: {url} — {e}')
        return None

# Download all filings with rate limiting (SEC allows 10 req/s)
filing_paths = []
failed = 0
for company in tqdm(companies, desc='Collecting filings'):
    filings = get_recent_filings(company['cik'], count=YEARS_PER_COMPANY)
    for f in filings:
        p = download_filing(f, EDGAR_DIR)
        if p:
            filing_paths.append({'path': str(p), 'ticker': company['ticker'], **f})
        else:
            failed += 1
        time.sleep(0.12)  # ~8 req/s to stay under SEC limit

print(f'\nDownloaded: {len(filing_paths)}, Failed: {failed}')

In [ ]:
# Cell 5: Parse financial statement tables from HTML filings

STATEMENT_KEYWORDS = {
    'income_statement': ['income', 'operations', 'earnings', 'revenue', 'profit and loss'],
    'balance_sheet': ['balance sheet', 'financial position', 'assets', 'liabilities'],
    'cash_flow': ['cash flow', 'cash and cash equivalents'],
}

def extract_tables_from_html(html_path: Path) -> list[dict]:
    """Extract financial tables from a 10-K HTML filing."""
    soup = BeautifulSoup(html_path.read_bytes(), 'lxml')
    tables = soup.find_all('table')
    results = []
    for i, table in enumerate(tables):
        rows = []
        for tr in table.find_all('tr'):
            cells = [td.get_text(strip=True) for td in tr.find_all(['td', 'th'])]
            if any(c for c in cells):
                rows.append(cells)
        if len(rows) < 3:  # Skip trivial tables
            continue
        # Classify table type based on surrounding text
        context = table.find_previous(string=True) or ''
        table_type = 'unknown'
        ctx_lower = context.lower() if context else ''
        for stype, keywords in STATEMENT_KEYWORDS.items():
            if any(kw in ctx_lower for kw in keywords):
                table_type = stype
                break
        results.append({
            'table_index': i,
            'type': table_type,
            'rows': rows,
            'num_rows': len(rows),
            'num_cols': max(len(r) for r in rows) if rows else 0,
        })
    return results

# Test parsing
if filing_paths:
    sample_tables = extract_tables_from_html(Path(filing_paths[0]['path']))
    print(f'Found {len(sample_tables)} tables in first filing')
    for t in sample_tables[:5]:
        print(f"  Table {t['table_index']}: {t['type']} ({t['num_rows']}×{t['num_cols']})")

In [ ]:
# Cell 6: Process all filings and save structured dataset
all_tables = []
for fp in tqdm(filing_paths, desc='Parsing HTML tables'):
    tables = extract_tables_from_html(Path(fp['path']))
    for t in tables:
        t['ticker'] = fp['ticker']
        t['cik'] = fp['cik']
        t['filing_date'] = fp['date']
    all_tables.extend(tables)

# Save dataset
out_file = EDGAR_DIR / 'extracted_tables.json'
with open(out_file, 'w') as f:
    json.dump(all_tables, f, indent=2)

print(f'Total tables extracted: {len(all_tables)}')
print(f'By type:')
from collections import Counter
for t, c in Counter(t['type'] for t in all_tables).most_common():
    print(f'  {t}: {c}')
print(f'Saved to: {out_file}')